In [1]:
#@title Import

import torch , torch.nn as nn , torch.nn.functional as F , torchvision
from torch.utils.data import DataLoader
from torchvision import transforms , datasets, models
import numpy as np
import matplotlib.pyplot as plt


Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
Device type: tpu, Device index: default
Initializing TPU distributed runtime


In [2]:
#@title Device

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [3]:
#@title Variables

BATCH_SIZE = 256
IMG_SIZE = 28
PATCH_SIZE = 7
ATTENTION_HEADS = 4
TRANSFORMER_LAYER = 4
EMBED_DIM = 32
CLASSES = 10
EPOCHS_STUDENT = 5
LR_STUDENT = 3e-4
CHANNELS = 3 # since our cnn require 3 channels so we expand out 1 channel input to 3 channel
TEMPERATURE = 2
ALPHA = 0.5

In [4]:
#@title Import the Data
data_transformations = transforms.Compose([transforms.ToTensor(),
                                           transforms.Lambda(lambda t: t.repeat(3,1,1))
                                           ])

train_dataset = datasets.MNIST('./data',train=True, download=True, transform= data_transformations)
val_dataset = datasets.MNIST('./data',train=False, download=True,  transform= data_transformations)

100%|██████████| 9.91M/9.91M [00:00<00:00, 18.5MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 497kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.73MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 13.2MB/s]


In [5]:
#@title Create train and val batches

train_dataloader = DataLoader(train_dataset,batch_size= BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(val_dataset,batch_size= BATCH_SIZE)

In [6]:
#@title Teacher Model

teacher = models.resnet50(weights= models.ResNet50_Weights.IMAGENET1K_V2)
# we have to remove last layer i.e 1000 class classification bcs we want it to predict just 10 class of mnist dataset
teacher.fc = nn.Linear(teacher.fc.in_features,CLASSES)
teacher.to(device)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 338MB/s]


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Con

In [7]:
#@title Student Vit Patch Embedding

# Patch Embed
# here kernel size and stride both are patch_size (suppose our patch size = 7 then our each patch will 7x7
# so whenever we want it to shift we have shift it to the same size so stride should also be 7x7 so that there would be no overlapping)
class PatchEmbed(nn.Module):
  def __init__(self,num_channels= CHANNELS, embed_dim=EMBED_DIM, patch_size= PATCH_SIZE,img_size = IMG_SIZE):
    super().__init__()
    self.proj = nn.Conv2d(num_channels ,embed_dim ,patch_size , patch_size)
    self.n = (img_size // patch_size) ** 2 # --> no.of patches or patch_num

  def forward(self,x):
    x = self.proj(x)
    x = x.flatten(2)
    x = x.transpose(1,2)
    return x


In [8]:
#@title Student Vit Class

# qkv_dim is nothing but dimension of each individual attention head
# * is used to repeat the line of code
# head_cls & head_distil is similar as the mlp head class we had in vit file
# they both are going used by the model to predict and calculate the loss
# BATCH_SIZE may varies for some batches in the end so its good to extract it first
class ViT(nn.Module):
  def __init__(self,embed_dim= EMBED_DIM, attention_heads= ATTENTION_HEADS, layers= TRANSFORMER_LAYER,classes = CLASSES):
    super().__init__()
    self.patch_embed = PatchEmbed()
    self.cls = nn.Parameter(torch.zeros(1,1,embed_dim))
    self.distillation = nn.Parameter(torch.zeros(1,1,embed_dim))
    n = self.patch_embed.n
    self.position = nn.Parameter(torch.zeros(1,n+2,embed_dim))
    qkv_dim = embed_dim // attention_heads
    self.transformer_block = nn.Sequential(*[nn.TransformerEncoderLayer(embed_dim,attention_heads,qkv_dim , batch_first=True) for _ in range(layers)])

    self.layernorm = nn.LayerNorm(embed_dim)
    self.head_cls = nn.Linear(embed_dim, classes)
    self.head_distil = nn.Linear(embed_dim, classes)

  def forward(self,x):
    B = x.shape[0]
    x = self.patch_embed(x)
    cls = self.cls.expand(B,-1,-1)
    dist = self.distillation.expand(B,-1,-1)
    x = torch.cat([cls,x,dist],1) + self.position # cls + 16 patches + dist, dim = 1 bcs 0 is batch_num and 1 is num_patches and we want out cls to concat in num_patches
    x = self.transformer_block(x)
    x = self.layernorm(x)

    cls_token = x[:,0] # for every batch 1 token i.e first token
    distil_token = x[:,-1] # for every batch 1 token i.e last token

    cls_logits = self.head_cls(cls_token)
    dist_logits = self.head_distil(distil_token)

    return cls_logits, dist_logits


In [9]:
#@title Define Student Model

student = ViT().to(device)
opt_s = torch.optim.AdamW(student.parameters(), lr= LR_STUDENT)

In [10]:
#@title Define the loss function
# kl divergence loss
def kd_loss(cls_logits, distil_logits, t_logits, y, alpha=ALPHA, temperature = TEMPERATURE):
  kd = F.kl_div(F.log_softmax(distil_logits/temperature, dim=1),
                F.softmax(t_logits/temperature, dim=1),
                reduction = 'batchmean') * (temperature**2)
  ce = F.cross_entropy(cls_logits, y)
  return alpha * kd + (1-alpha) * ce

In [11]:
from math import dist
#@title Run Epochs
for e in range(EPOCHS_STUDENT):
  student.train()
  for x,y in train_dataloader:
    x,y = x.to(device), y.to(device)
    with torch.no_grad(): t_logits = teacher(x)
    cls_logits, dist_logits = student(x)
    loss = kd_loss(cls_logits, dist_logits, t_logits, y)
    opt_s.zero_grad(); loss.backward(); opt_s.step()
  print(f"Epochs {e+1} done")

Epochs 1 done
Epochs 2 done
Epochs 3 done
Epochs 4 done
Epochs 5 done


In [12]:
#@title Evaluate
student.eval()
correct=0; total=0
samples=[]
with torch.no_grad():
  for x,y in val_dataloader:
    x,y = x.to(device), y.to(device)
    cls_logits, dist_logits = student(x)
    cls_dist = (cls_logits + dist_logits)/2
    pred = cls_dist.argmax(1)
    correct += (pred==y).sum().item(); total += y.size(0)
    if len(samples)<5:
      samples.append((x.cpu(),pred.cpu(),y.cpu()))

In [13]:
#@title Accuracy
acc = 100 * correct/total
print(f"Test accuracy: {acc:.2f}%")

Test accuracy: 90.45%
